# 실습 04 · UCI HAR: 1D CNN과 LSTM 공정 비교 · QUIZ

이 노트북은 tutorial을 완료한 후 직접 실습하는 것입니다.
**문제만 주어집니다.** 각 문제를 풀고 정답과 비교하세요.

**시간:** 약 30분  
**평가:** 센서 데이터 처리, Conv1d vs LSTM 이해, 공정 비교

**⭐ 푸는 방법:** 각 코드 셀에 뼈대 코드가 주어집니다. `____` 부분만 채운 뒤 실행하세요.
막히면 tutorial의 해당 섹션을 참고해도 됩니다 — 그것도 정식 방법입니다.

## 환경 설정 및 데이터 로드

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from pathlib import Path

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("data 폴더를 찾지 못했습니다. 노트북과 같은 위치(또는 상위)에 data 폴더가 있어야 합니다.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
HAR_ROOT = DATA_ROOT / "UCI HAR Dataset"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SIGNAL_NAMES = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]
CLASS_NAMES = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]
print("device:", DEVICE)

## 문제 1: HAR 데이터 읽기

다음을 수행하세요:
1. 9개의 센서 신호(signal)를 HAR_ROOT/train/Inertial Signals에서 로드하세요.
2. 각 신호를 stack하여 (N, 9, 128) shape로 만드세요.
3. label을 읽고 1을 빼서 0-5 범위로 조정하세요.
4. train과 test의 shape를 출력하세요.

**예상:**
- train: (2400, 9, 128) — 수업용 축소본 기준
- test: (950, 9, 128)

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
def load_har_split(split):
    signal_dir = HAR_ROOT / split / "Inertial Signals"
    channels = [np.loadtxt(signal_dir / f"{name}_{split}.txt", dtype=np.float32) for name in SIGNAL_NAMES]
    X = np.____(channels, axis=____)     # 9개 표를 채널 축으로 쌓기 → (N, 9, 128)
    y = np.loadtxt(HAR_ROOT / split / f"y_{split}.txt", dtype=np.int64) - ____   # 1~6 → 0~5
    return X, y

X_official_train, y_official_train = load_har_split("train")
X_official_test, y_official_test = load_har_split("____")

print("train:", X_official_train.shape)   # 수업용 축소본: (2400, 9, 128)
print("test :", X_official_test.shape)    # (950, 9, 128)

## 문제 2: 활동별 sample 수 확인

다음을 수행하세요:
1. train/test에서 각 활동별 sample 수를 세세요.
2. DataFrame으로 표현하고 출력하세요.
3. class 분포가 균형잡혀 있는지 확인하세요.

**목표:** accuracy 해석 시 특정 활동이 과도하게 많지 않은지 확인

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
train_counts = np.____(y_official_train, minlength=____)   # 번호별 개수 세기 / 행동 종류 수
test_counts = np.bincount(y_official_test, minlength=6)
count_table = pd.DataFrame({"activity": CLASS_NAMES, "train": train_counts, "test": test_counts})
display(count_table)   # 쏠림 없이 균형인지 확인

## 문제 3: train/validation/test 분할

다음을 수행하세요:
1. train을 80% train, 20% validation으로 분할하세요 (stratified).
2. CPU 수업 시간을 위해 subset을 만드세요:
   - train을 1800개, validation을 450개, test를 900개로 제한하세요.
   - `stratified_limit` 함수를 구현하거나 `train_test_split`을 다시 사용하세요.
3. 각 subset의 class 분포를 확인하세요.

**왜:** CPU 학습을 빠르게 하면서 class 균형을 유지

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
all_idx = np.arange(len(y_official_train))
train_idx, val_idx = train_test_split(all_idx, test_size=____, stratify=____, random_state=SEED)   # 20% / 비율 유지 기준

def stratified_limit(indices, labels, limit, seed):
    if len(indices) <= limit:
        return indices
    chosen, _ = train_test_split(indices, train_size=limit, stratify=labels[indices], random_state=seed)
    return np.sort(chosen)

train_idx = stratified_limit(train_idx, y_official_train, ____, SEED)    # 1800건으로 제한
val_idx = stratified_limit(val_idx, y_official_train, 450, SEED)
test_idx = stratified_limit(np.arange(len(y_official_test)), y_official_test, 900, SEED)

X_train, y_train = X_official_train[train_idx], y_official_train[train_idx]
X_val, y_val = X_official_train[val_idx], y_official_train[val_idx]
X_test, y_test = X_official_test[test_idx], y_official_test[test_idx]
print("sizes:", X_train.shape, X_val.shape, X_test.shape)

## 문제 4: 채널별 표준화

다음을 수행하세요:
1. train 데이터의 채널별(축 0과 2) 평균과 표준편차를 계산하세요.
2. (X - mean) / std로 표준화하세요.
3. val과 test도 같은 scaler로 변환하세요.
4. train의 표준화된 평균이 ~0인지 확인하세요.

**참고:** 두 모델은 **같은 전처리된 데이터**를 공유합니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
channel_mean = X_train.mean(axis=(0, ____), keepdims=True)   # 건수·시간 방향을 평균으로 없애기
channel_std = X_train.____(axis=(0, 2), keepdims=True) + 1e-6   # 폭(표준편차) 계산

X_train = ((X_train - channel_mean) / channel_std).astype(np.float32)
X_val = ((X_val - ____) / channel_std).astype(np.float32)    # 같은 기준으로!
X_test = ((X_test - channel_mean) / channel_std).astype(np.float32)

print("train channel mean:", np.round(X_train.mean(axis=(0, 2)), 3))   # 전부 0 근처여야 합니다

## 문제 5: 공유 DataLoader

다음을 수행하세요:
1. TensorDataset으로 train/val/test dataset을 만드세요.
2. DataLoader로 변환하세요 (batch_size=128, train은 shuffle=True).
3. 첫 번째 batch의 shape를 확인하세요.

**참고:** `make_loaders(seed)` 함수를 구현하면 두 모델에서 재사용할 수 있습니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_dataset = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

def make_loaders(seed):
    generator = torch.Generator().manual_seed(seed)
    return (
        DataLoader(train_dataset, batch_size=128, shuffle=____, generator=generator),
        DataLoader(val_dataset, batch_size=128, shuffle=False),
        DataLoader(test_dataset, batch_size=128, shuffle=False),
    )

train_loader, val_loader, test_loader = make_loaders(SEED)
xb, yb = next(iter(____))             # 학습용 loader에서 한 batch
print("batch:", xb.shape, yb.shape)   # (128, 9, 128), (128,)

## 문제 6: 1D CNN 모델 구현

다음 구조로 Conv1d 기반 HAR 모델을 구현하세요:

```python
class HARConv1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Conv1d(9, 16, kernel=5, padding=2) + ReLU + MaxPool1d(2)
            # Conv1d(16, 32, kernel=5, padding=2) + ReLU
            # AdaptiveAvgPool1d(1) - sequence를 단일 값으로
        )
        self.classifier = nn.Linear(32, 6)
    
    def forward(self, x):
        # x: (N, 9, 128)
        # 출력: (N, 6)
```

**설명:** Conv1d는 시간축을 따라 이동하는 커널로 지역 패턴을 찾습니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
class HARConv1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(____, 16, kernel_size=5, padding=2),   # 입력 채널(센서) 수
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Linear(32, ____)   # 행동 종류 수

    def forward(self, x):
        return self.classifier(self.features(x).____(-1))   # 길이 1이 된 마지막 축 제거

cnn = HARConv1D().to(DEVICE)
print("CNN parameters:", sum(p.numel() for p in cnn.parameters()))

## 문제 7: LSTM 모델 구현

다음을 수행하세요:

```python
class HARLSTM(nn.Module):
    def __init__(self, hidden_size=24):
        super().__init__()
        self.lstm = nn.LSTM(input_size=9, hidden_size=hidden_size, batch_first=True)
        self.classifier = nn.Linear(hidden_size, 6)
    
    def forward(self, x):
        # x: (N, 9, 128)
        # transpose → (N, 128, 9)
        # LSTM 통과 → 마지막 hidden state 사용
        # 출력: (N, 6)
```

**주의:** LSTM은 입력 shape가 (N, sequence_length, features)입니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
class HARLSTM(nn.Module):
    def __init__(self, hidden_size=24):
        super().__init__()
        self.lstm = nn.LSTM(input_size=____, hidden_size=hidden_size, batch_first=____)   # 한 시점의 값 개수 / batch가 앞에 오는 형식
        self.classifier = nn.Linear(hidden_size, 6)

    def forward(self, x):
        sequence = x.____(1, 2)                 # (N,9,128) → (N,128,9) 축 교환
        _, (hidden, _) = self.lstm(sequence)
        return self.classifier(hidden[____])    # 마지막 층의 hidden state

lstm = HARLSTM().to(DEVICE)
print("LSTM parameters:", sum(p.numel() for p in lstm.parameters()))

## 문제 8: 공통 학습·검증 함수

다음을 수행하세요:
1. `run_epoch(model, loader, criterion, optimizer=None)` 함수를 구현하세요.
2. training 중에만 gradient 계산하세요.
3. (평균 loss, 정확도)를 반환하세요.

**참고:** 이전 quiz와 유사합니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not ____        # optimizer를 받았으면 학습 모드
    model.train(____)                       # 모델에 현재 모드를 알림
    total_loss = total_correct = total_count = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                loss.____()                 # 수정 방향 계산
                optimizer.____()            # 수정 실행
        total_loss += loss.item() * len(yb)
        total_correct += (logits.____(1) == yb).sum().item()   # 점수가 가장 높은 선택지 번호
        total_count += len(yb)
    return total_loss / total_count, total_correct / total_count

print("run_epoch 정의 완료")

## 문제 9: 두 모델 학습 및 비교

다음을 수행하세요:
1. CNN 모델을 생성하고 10 epoch 학습하세요.
2. LSTM 모델을 생성하고 10 epoch 학습하세요.
3. **같은 DataLoader를 사용하세요** (같은 seed).
4. 각 epoch마다 train/validation accuracy를 출력하세요.
5. test accuracy를 평가하세요.

**학습 조건:**
- criterion: CrossEntropyLoss()
- optimizer: Adam(lr=0.001)
- epochs: 10

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
criterion = nn.CrossEntropyLoss()

# --- CNN 학습 ---
torch.manual_seed(SEED)
cnn = HARConv1D().to(DEVICE)
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)
train_loader, val_loader, test_loader = make_loaders(SEED)
for epoch in range(1, 11):
    tr_loss, tr_acc = run_epoch(cnn, train_loader, criterion, optimizer)
    va_loss, va_acc = run_epoch(cnn, val_loader, criterion)
    if epoch in {1, 5, 10}:
        print(f"CNN epoch {epoch} | train {tr_acc:.3f} | val {va_acc:.3f}")
cnn_test_loss, cnn_test_acc = run_epoch(cnn, ____, criterion)   # 최종 평가용 loader
print(f"CNN test accuracy: {cnn_test_acc:.3f}")

# --- LSTM 학습: 같은 조건으로 (무엇을 같게 해야 공정 비교일까요?) ---
torch.manual_seed(SEED)
lstm = HARLSTM().to(DEVICE)
optimizer = torch.optim.Adam(lstm.parameters(), lr=____)      # CNN과 같은 값
train_loader, val_loader, test_loader = make_loaders(____)    # CNN과 같은 seed
for epoch in range(1, 11):
    tr_loss, tr_acc = run_epoch(lstm, train_loader, criterion, optimizer)
    va_loss, va_acc = run_epoch(lstm, val_loader, criterion)
    if epoch in {1, 5, 10}:
        print(f"LSTM epoch {epoch} | train {tr_acc:.3f} | val {va_acc:.3f}")
lstm_test_loss, lstm_test_acc = run_epoch(lstm, test_loader, criterion)
print(f"LSTM test accuracy: {lstm_test_acc:.3f}")

## 문제 10: 활동별 정확도 분석

다음을 수행하세요:
1. test 데이터에서 각 모델의 예측값을 얻으세요.
2. 각 활동별로 정확도를 계산하세요 (recall 기준).
3. 두 모델이 어느 활동에서 잘하고 못하는지 비교하세요.
4. 활동별 정확도를 DataFrame으로 출력하세요.

**설명:** 전체 accuracy 하나보다 "어느 활동이 어려운가"를 아는 것이 중요합니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
def collect_predictions(model, loader):
    model.eval()
    true_values, predictions = [], []
    with torch.no_grad():
        for xb, yb in loader:
            predictions.extend(model(xb.to(DEVICE)).____(1).cpu().tolist())   # 점수 최고 선택지 번호
            true_values.extend(yb.tolist())
    return np.array(true_values), np.array(predictions)

_, _, shared_test_loader = make_loaders(SEED)
cnn_true, cnn_pred = collect_predictions(cnn, shared_test_loader)
lstm_true, lstm_pred = collect_predictions(lstm, shared_test_loader)

per_class = []
for class_id, name in enumerate(CLASS_NAMES):
    mask = cnn_true == ____   # 이 행동에 해당하는 데이터만 고르는 조건
    per_class.append({
        "activity": name,
        "CNN": (cnn_pred[mask] == class_id).mean(),
        "LSTM": (lstm_pred[mask] == class_id).mean(),
    })
display(pd.DataFrame(per_class).round(3))   # 두 모델이 어느 행동에서 갈리나요?